**Day 1 - Setup & Parser:**

Same task as Feature 1.Checkpoint:print total messages parsed(should be 3,174 real messages), plus the first 5 and 5 parsed messages to verify the parser works correctly.

**Feature 1 - The Chat Parser:**

Read hostel_bois.txt,split into lines,and extract timestamp,sender and message text from each real message.Skip system messages;separately count media-omitted and deleted messages.Store everything as a list of dicts.

In [22]:
#Feature 1 : The Chat Parser
#Reads hostel_bois.txt, classifies each line and extracts
#timestamp, sender and message text for real messages only

#Step 1 : Read the file
with open('hostel_bois.txt','r',encoding='utf-8') as f:
    lines=f.read().split('\n')

#Step 2 : Set up counters and storage
messages=[]         #List of dicts:{'timestamp','sender','text'}
system_count=0
media_count=0
deleted_count=0

#Step 3 : Helper to check if a line starts with a date pattern
def starts_with_date(line):
  #Checks first 8 characters look like DD/MM/YY
  if len(line) < 8:
    return False
  date_part=line[:8]
  return (date_part[0:2].isdigit() and date_part[2] == '/'and
          date_part[3:5].isdigit() and date_part[5] == '/' and
          date_part[6:8].isdigit())

#Step 4 : Loop through every line
for line in lines:
  line=line.strip()

  #Skip empty lines
  if line == '':
    continue

  #Multi-line message continuation (not in this dataset, but handled anyway)
  if not starts_with_date(line):
    if messages:
        messages[-1]['text'] += '\n' + line
    continue

  # Split timestamp from the rest:"DD/MM/YY, HH:MM - <rest>"
  parts=line.split('-',1)
  if len(parts) != 2:
     system_count += 1
     continue

  timestamp,rest=parts

  # a. System messages have no "sender: message" structure
  if ':' not in rest:
    system_count += 1
    continue

  sender,text=rest.split(':',1)

  # b. Media omitted - count as a message,but flag it
  if text.strip() == '<Media omitted>':
     media_count += 1
     messages.append({'timestamp':timestamp,'sender':sender,'text':text,'type':'media'})
     continue

  # c. Deleted messages
  if text.strip() == 'This message was deleted':
   deleted_count += 1
   messages.append({'timestamp':timestamp,'sender':sender,'text':text,'type':'deleted'})
   continue

  # Real message
  messages.append({'timestamp':timestamp,'sender':sender,'text':text,'type':'real'})

#Step 5 : Checkpoint output
real_messages=[m for m in messages if m['type'] == 'real']
senders=set(m['sender'] for m in real_messages)

print(f"Successfully parsed {len(messages)} from {len(senders)} participants.")
print(f"Skipped {system_count} system messages.")
print(f"Media-ommited: {media_count}, Deleted: {deleted_count}")
print(f"Real messages: (used for text analysis): {len(real_messages)}")

print("\n-- First 5 parsed messages --")
for m in messages[:5]:
  print(m)

print("\n-- Last 5 parsed messages --")
for m in messages[-5:]:
  print(m)

Successfully parsed 3174 from 6 participants.
Skipped 4 system messages.
Media-ommited: 32, Deleted: 15
Real messages: (used for text analysis): 3127

-- First 5 parsed messages --
{'timestamp': '01/04/24, 01:17 ', 'sender': ' Rahul', 'text': ' scene fix', 'type': 'real'}
{'timestamp': '01/04/24, 01:17 ', 'sender': ' Rahul', 'text': ' haan', 'type': 'real'}
{'timestamp': '01/04/24, 01:18 ', 'sender': ' Rahul', 'text': ' kya scene', 'type': 'real'}
{'timestamp': '01/04/24, 02:13 ', 'sender': ' Rahul', 'text': ' abhi free hai?', 'type': 'real'}
{'timestamp': '01/04/24, 02:13 ', 'sender': ' Rahul', 'text': ' abey', 'type': 'real'}

-- Last 5 parsed messages --
{'timestamp': '30/05/24, 19:14 ', 'sender': ' Priya', 'text': ' Take care everyone', 'type': 'real'}
{'timestamp': '30/05/24, 19:28 ', 'sender': ' Priya', 'text': ' Karan that sounds tough, take care', 'type': 'real'}
{'timestamp': '30/05/24, 21:17 ', 'sender': ' Aman', 'text': ' the existential dread is back', 'type': 'real'}
{'tim

**Day 2 - Group Overview & Messages Per Person:**

The Overview numbers should match Section 4 of the Brief(3,174 messages,6 participants,60-day period).

**Feature 2 - Group Overview**

Stats - total messages,date range(first to last message date),total days,participant count and per-person message counts sorted highest to lowest,

In [23]:
#Feature 2 : Group Overview
#Computes total messages,date range,participants,
#and per-person message counts sorted highest to lowest.

from datetime import datetime

#Step 1 : Count real + media + deleted messages (person did send something)
countable_messages=[]
for m in messages:
  if m['type'] == 'real' or m['type'] == 'media' or m['type'] == 'deleted':
    countable_messages.append(m)

#Step 2 : Count messages per person
person_counts={}
for m in countable_messages:
     sender=m['sender']
     if sender in person_counts:
        person_counts[sender] = person_counts[sender]+1
     else:
        person_counts[sender]=1

#Sort people by count,highest first
sorted_people=sorted(person_counts.items(),key=lambda x:x[1],reverse=True)

#Step 3 : Date range (clean date and time separately, then rebuild)
all_dates=[]
for m in countable_messages:
  ts=m['timestamp']

  date_part,time_part=ts.split(',',1)

  clean_ts=""
  for ch in ts:
    if ch.isdigit() or ch in "/, :":
      clean_ts=clean_ts+ch
  clean_ts=clean_ts.strip()

  dt=datetime.strptime(clean_ts,'%d/%m/%y, %H:%M')
  all_dates.append(dt)

first_date=min(all_dates)
last_date=max(all_dates)
total_days=(last_date.date() - first_date.date()).days+1

#Step 4 : Totals
total_messages=len(countable_messages)
total_participants=len(person_counts)

#Step 5 : Print the Group Overview
print("=" * 60)
print("GROUP OVERVIEW")
print("=" * 60)
print("Group          : Hostel Bois 4ever")
print("Period         : ",first_date.strftime('%d %B %Y'), "to", last_date.strftime('%d %b %Y'), f"({total_days} days)")
print("Total messages : ",total_messages)
print("Participants   : ",total_participants)
print()
print("MESSAGES PER PERSON")
for person, count in sorted_people:
  percentage=(count/total_messages)*100
  print(f"{person:<10}: {count:>5} ({percentage:>5.1f}%)")

GROUP OVERVIEW
Group          : Hostel Bois 4ever
Period         :  01 April 2024 to 30 May 2024 (60 days)
Total messages :  3174
Participants   :  6

MESSAGES PER PERSON
 Rahul    :   953 ( 30.0%)
 Priya    :   718 ( 22.6%)
 Neha     :   635 ( 20.0%)
 Aman     :   490 ( 15.4%)
 Karan    :   354 ( 11.2%)
 Vikas    :    24 (  0.8%)


**Feature 3 - Most Active Day and Hour:**

Find the single day (across all 60 days) with the most messages,and the hour of the day(combined across all days) with the highest message volume.

In [24]:
#Feature 3 : Most Active Day and Hour
#Finds the single busiest day and the busiest hour across all 60 days.

#Step 1 : Count messages per date, and per hour
day_counts={}
hour_counts={}

for m in countable_messages:
  ts=m['timestamp']
  date_part,time_part=ts.split(',',1)

  #Clean date part (keep digits and '/')
  clean_date=""
  for ch in date_part:
    if ch.isdigit() or ch == '/':
      clean_date=clean_date+ch

  #Clean time part (keep digits and ':')
  clean_time=""
  for ch in time_part:
   if ch.isdigit() or ch == ':':
        clean_time=clean_time+ch

  #Extract hour (first 2 charaters of clean_time, e.g. "23:14" -> "23")
  hour=clean_time.split(':',1)[0]

  #Step 2 : Count messages for this date
  if clean_date in day_counts:
    day_counts[clean_date]=day_counts[clean_date]+1
  else:
    day_counts[clean_date]=1

  #Step 3 : Count messages for this hour
  if hour in hour_counts:
    hour_counts[hour]=hour_counts[hour]+1
  else:
    hour_counts[hour]=1

#Step 4 : Find the busiest day
busiest_day=max(day_counts,key=day_counts.get)
busiest_day_count=day_counts[busiest_day]

#Step 5 : Find the busiest hour
busiest_hour=max(hour_counts,key=hour_counts.get)
busiest_hour_count=hour_counts[busiest_hour]

#Convert busiest_day from "14/04/24" to  a readable format
from datetime import datetime
busiest_day_dt=datetime.strptime(busiest_day,'%d/%m/%y')
busiest_day_readable=busiest_day_dt.strftime('%d %B %Y')

#Format busiest hour as a range, e.g. "17.00-18.00"
busiest_hour_int=int(busiest_hour)
next_hour_int=(busiest_hour_int+1)%24
busiest_hour_range=f"{busiest_hour_int:02d}.00-{next_hour_int:02d}.00"

#Step 6 : Print the result
print("Busiest day : ",busiest_day_readable, f"({busiest_day_count}messages)")
print("Busiest hour: ",busiest_hour_range,f"({busiest_hour_count}messages)")


Busiest day :  04 May 2024 (76messages)
Busiest hour:  18.00-19.00 (248messages)


**Day 3 - Top Words:**

Top 10 most often used words in the group chat.

**Feature 5  - Word Frequency**

Extract every word from real messages,count frequency(lowercase,strip punctuayion,skip stop words).

In [25]:
#Feture 5 : Top Words with Frequency
#Counts how often each word appears across all real messages,
#and displays the top 10 with a bar proportional to its count.

#Step 1 : Get only real messages(exclude system/media/deleted)
real_messages=[]
for m in messages:
  if m['type']=='real':
    real_messages.append(m)

#Step 2 : Stop words to ignore(common English filler words)
stop_words=['i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
              'you', 'it', 'this', 'that', 'are', 'was', 'be', 'have', 'has',
              'my', 'your', 'we', 'he', 'she', 'they', 'do', 'does', 'did',
              'how', 'so', 'about', 'am', 'today', 'at', 'his', 'just', 'not',
              'if', 'with', 'can', 'will', 'what', 'when', 'why', 'who', 'all',
              'but', 'no', 'yes', 'me', "i'm", 'im', 'u', 'ur', 'which',
              'everyone', 'from', 'up', 'one', 'had', 'started', 'entire',
              'please', 'anyone', 'now', 'everything', 'came', 'went', 'going',
              'get', 'got', 'more', 'than', 'then', 'also', 'even', 'being',
              'been', 'out', 'our', 'their', 'them', 'some', 'any', 'into',
              'over', 'still', 'because', 'would', 'could', 'should', 'there',
              'here', 'were', 'as', 'by', "it's", 'its', 'while', 'during',
              'after', 'before', 'off']

#Step 3 : Punctuation characters strip from each word
punctuation='.,!?"\'()[]{}:;-_'

#Step 4 : Count every word
word_counts={}

for m in real_messages:
  text=m['text']
  words=text.split()

  for word in words:
    word=word.lower()
    cleaned_word=word.strip(punctuation)

    if cleaned_word=='':
      continue
    if cleaned_word in stop_words:
      continue

    #Skip empty strings and stop words
    if cleaned_word in word_counts:
      word_counts[cleaned_word]=word_counts[cleaned_word]+1
    else:
      word_counts[cleaned_word]=1

#Step 5 : Sort ALL words by count, highest first
sorted_words=sorted(word_counts.items(),key=lambda x:x[1],reverse=True)
top_10_words=sorted_words[:10]

#Step 6 : Print each  word with a bar sized to its count
print("THIS GROUP'S FAVOURITE WORDS")
print()

#Find the max count to scale bar length
max_count=top_10_words[0][1]

for word,count in top_10_words:
  bar_length=int((count/max_count)*20)
  bar='█' * bar_length
  print(f"{word:<10}:{bar} {count}")


THIS GROUP'S FAVOURITE WORDS

guys      :████████████████████ 318
hai       :████████████████ 268
telling   :███████████ 179
bhai      :██████████ 160
scene     :█████████ 145
yaar      :████████ 139
kya       :████████ 133
sleep     :███████ 112
ja        :██████ 101
raha      :██████ 101


**Day 4 - NumPy Activity Heatmap:**

Turn each person's hourly message counts into a NumPy matrix,then print it as a block-character heatmap.


**Feature 4 - Activity Heatmap:**

Build a 6*24 NumPy matrix(people * hours) counting each person's messages per hour,then render it as a text-based heatmap using 4 shADING LEVELS (. ░ ▒ █).

In [26]:
#Feature 4 : Activity Heatmap(NumPy)
#Builds a 6*24 matrix : rows = participants,columns=hours(0-23)
#Each cell=how many messages that person sent during that hour.

import numpy as np

#Step 1 : Get list of participants(order matters for the matrix)
participants=[]
for m in messages:
  if m['type'] in ('real', 'media', 'deleted'):
    if m['sender'] not in participants:
      participants.append(m['sender'])

num_people=len(participants)

#Step 2 : Create the 6*24 matrix,filled with zeros
heatmap=np.zeros((num_people,24))

#Step 3 : Fill the matrix by looping through every message
for m in messages:
  if m['type']  not in ('real', 'media', 'deleted'):
    continue

  sender=m['sender']
  ts=m['timestamp']

  #Clean and extract the hour from the timestamp
  date_part,time_part=ts.split(',',1)
  clean_time=""
  for ch in time_part:
    if ch.isdigit() or ch == ':':
      clean_time=clean_time+ch
  hour=int(clean_time.split(':')[0])

  person_index=participants.index(sender)
  heatmap[person_index][hour]=heatmap[person_index][hour]+1

#Step 4 : Render the heatmap using block characters
#Group hours into 8 blocks of 3 hours each,for a compact display
print("ACTIVITY HEATMAP (messages by hour)")
print("     00  03  06  09  12  15  18  21")

for i in range(num_people):
  person=participants[i]
  row=heatmap[i]
  person_max=row.max()

  line=f" {person:<8}"

  #Sum every 3-hour block for display(0-2,3-5...,21-23)
  for block_start in range(0,24,3):
    block_sum=row[block_start:block_start+3].sum()

    #Determine shading level relative to this person's max hourly count
    if person_max==0:
      symbol=". "
    else:
      ratio=block_sum/(person_max*3)
      if ratio <= 0.25:
        symbol=". "
      elif ratio <= 0.50:
        symbol="░"
      elif ratio <= 0.75:
        symbol="▒"
      else:
        symbol="█"

    line=line+symbol

    print(line)

#Step 5 : Quick NumPy aggregation check
print()
print("Total messages per person(from heatmap):")
for i in range(num_people):
  print(f"{participants[i]:<10}: {int(heatmap[i].sum())}")

ACTIVITY HEATMAP (messages by hour)
     00  03  06  09  12  15  18  21
  Rahul  . 
  Rahul  . . 
  Rahul  . . . 
  Rahul  . . . . 
  Rahul  . . . . ░
  Rahul  . . . . ░▒
  Rahul  . . . . ░▒▒
  Rahul  . . . . ░▒▒▒
  Priya  . 
  Priya  . . 
  Priya  . . ░
  Priya  . . ░█
  Priya  . . ░██
  Priya  . . ░██▒
  Priya  . . ░██▒▒
  Priya  . . ░██▒▒░
  Karan  . 
  Karan  . . 
  Karan  . . . 
  Karan  . . . ░
  Karan  . . . ░█
  Karan  . . . ░█▒
  Karan  . . . ░█▒▒
  Karan  . . . ░█▒▒░
  Neha   . 
  Neha   . . 
  Neha   . . ░
  Neha   . . ░▒
  Neha   . . ░▒▒
  Neha   . . ░▒▒▒
  Neha   . . ░▒▒▒█
  Neha   . . ░▒▒▒█░
  Aman   ▒
  Aman   ▒▒
  Aman   ▒▒. 
  Aman   ▒▒. . 
  Aman   ▒▒. . . 
  Aman   ▒▒. . . . 
  Aman   ▒▒. . . . . 
  Aman   ▒▒. . . . . ░
  Vikas  . 
  Vikas  . . 
  Vikas  . . ░
  Vikas  . . ░. 
  Vikas  . . ░. ░
  Vikas  . . ░. ░▒
  Vikas  . . ░. ░▒▒
  Vikas  . . ░. ░▒▒░

Total messages per person(from heatmap):
 Rahul    : 953
 Priya    : 718
 Karan    : 354
 Neha     : 635
 Aman    

**Day 5 - Response Times:**

For each person,computing their average response time-the gap between someone else's message and their next reply.

**Feature 6 - Silent Streaks:**

For each person ,computing their longest silent streak-the longest run of consecutive days with 0 messages.

In [27]:
#Feature 6 : Response Speed & Silent Streaks
#(a) Average response time per person
#(b) Longest silent streak(consecutive days with zero messages)

from datetime import datetime,timedelta

#Step 1 : Build a clean list of (datetime,sender) for all countable messages
timed_messages=[]

for m in messages:
  if m['type'] not in ('real','media','deleted'):
      continue

  ts=m['timestamp']
  date_part,time_part=ts.split(',',1)

  clean_date=""
  for ch in date_part:
      if ch.isdigit() or ch == '/':
          clean_date=clean_date+ch

  clean_time=""
  for ch in time_part:
      if ch.isdigit() or ch == ':':
          clean_time=clean_time+ch

  clean_ts=clean_date+","+clean_time
  dt=datetime.strptime(clean_ts,'%d/%m/%y,%H:%M')

  timed_messages.append({'datetime':dt,'sender':m['sender']})

#Step 2 : Sort messages in chronological order
timed_messages=sorted(timed_messages,key=lambda x:x['datetime'])

#Step 3 : Compute response gaps
#A "response" happens when the sender changes from the previous message
response_gaps={}   #{person:[list of gaps in seconds]}

for i in range(1,len(timed_messages)):
    current=timed_messages[i]
    previous=timed_messages[i-1]

    if current['sender'] != previous['sender']:
        gap_seconds=(current['datetime']-previous['datetime']).total_seconds()
        person=current['sender']

        if person in response_gaps:
            response_gaps[person].append(gap_seconds)
        else:
            response_gaps[person]=[gap_seconds]

#Step 4 : Compute average response time per person(in minutes)
avg_response_minutes={}
for person in response_gaps:
    gaps=response_gaps[person]
    avg_seconds=sum(gaps)/len(gaps)
    avg_response_minutes[person]=avg_seconds/60

fastest_person=min(avg_response_minutes,key=avg_response_minutes.get)
slowest_person=max(avg_response_minutes,key=avg_response_minutes.get)

#Step 5 : Find longest silent streak per person
first_date=timed_messages[0]['datetime'].date()
last_date=timed_messages[-1]['datetime'].date()
total_days=(last_date-first_date).days+1

#Get list of unique participants:
participants=[]
for m in timed_messages:
    if m['sender'] not in participants:
       participants.append(m['sender'])

silent_streaks={} #{person: (max_streak,streak_start_date,streak_end_date)}

for person in participants:
    #Build set of dates this person was active
    active_dates=set()
    for m in timed_messages:
        if m['sender'] == person:
            active_dates.add(m['datetime'].date())

    max_streak=0
    max_streak_start=None
    max_streak_end=None
    current_streak=0
    current_streak_start=None

    for day_offset in range(total_days):
        current_date=first_date+timedelta(days=day_offset)

        if current_date not in active_dates:
            if current_streak==0:
                current_streak_start=current_date
            current_streak=current_streak+1

            if current_streak > max_streak:
                max_streak=current_streak
                max_streak_start=current_streak_start
                max_streak_end=current_date
        else:
            current_streak=0

    silent_streaks[person]=(max_streak,max_streak_start,max_streak_end)

#Step 6 : Print the results:
print("RESPONSE PATTERNS")
print(f"Fastest replier: {fastest_person} (avg{avg_response_minutes[fastest_person]:.1f}minutes)")

if avg_response_minutes[slowest_person] >= 60:
    print(f"Slowest replier: {slowest_person} (avg{avg_response_minutes[slowest_person]/60:.1f} hours)")
else:
    print(f"Slowest replier: {slowest_person} (avg{avg_response_minutes[slowest_person]:.1f} minutes)")

print()
print("LONGEST SILENT STREAKS (consecutive days with zero messages)")

#Sort people by streak length,highest first
sorted_streaks=sorted(silent_streaks.items(),key=lambda x:x[1][0],reverse=True)

for person,(streak,start,end) in sorted_streaks:
 if streak==0:
   print(f"{person:<10}: 0 days (never went silent)")
 else:
   print(f"{person:<10}: {streak} days ({start.strftime('%d %b')}-{end.strftime('%d %b')})")


RESPONSE PATTERNS
Fastest replier:  Rahul (avg34.9minutes)
Slowest replier:  Aman (avg55.4 minutes)

LONGEST SILENT STREAKS (consecutive days with zero messages)
 Vikas    : 11 days (23 Apr-03 May)
 Rahul    : 0 days (never went silent)
 Priya    : 0 days (never went silent)
 Karan    : 0 days (never went silent)
 Neha     : 0 days (never went silent)
 Aman     : 0 days (never went silent)


**Day 6 - Archetype Detection:**

One scoring function per archetype, then assign each person their best-scoring, exclusive archetype.

**Feature 7 - Personality Archetype:**

Assign each participant one of 8 personality archetypes based on quantitative rules(spam bursts, caring keywords, night activity %, words per message, all-caps %, silence %, funny-word %, question %). Each person gets exactly one archetype - the one they score highest on.

In [28]:
#Feature 7 : Personality Archetype Detection
#Scores every person on 8 archetypes,then assigns each person
#Step 1 : Group real messages by person(in chronological order)
person_messages={}     #{person:[list of message texts]}
for m in messages:
    if m['type'] =='real':
       sender=m['sender']
       if sender not  in person_messages:
           person_messages[sender]=[]
       person_messages[sender].append(m['text'])

participants=list(person_messages.keys())

#================================================================
#Archetype 1 : THE SPAMMER - avg consecutive message burst > 3
#================================================================
def score_spammer(person):
    bursts=[]
    current_burst=0
    last_sender=None

    for m in messages:
        if m['type'] not in('real', 'media', 'deleted'):
            continue
        if m['sender'] == person:
            if last_sender == person:
                current_burst=current_burst+1
            else:
                if current_burst > 0:
                    bursts.append(current_burst)
                current_burst=1
        else:
           if last_sender == person and current_burst > 0:
               bursts.append(current_burst)
               current_burst=0
        last_sender=m['sender']

    if current_burst > 0:
        bursts.append(current_burst)

    if len(bursts) == 0:
        return 0
    return sum(bursts)/len(bursts)

#=======================================================
#Archetype 2 : THE GROUP MOM - caring keyword count
#=======================================================
caring_keywords=['okay','safe','eat','sleep','take care','are you','please','reminder','drink water',"dont forget"]

def score_group_mom(person):
    score=0
    for text in person_messages.get(person,[]):
        text_lower=text.lower()
        for keyword in caring_keywords:
            if keyword in text_lower:
             score=score+1
    return score

#===============================================================
#Archetype 3 : THE NIGHT OWL - % messages between 23:00 - 04:59
#================================================================
def score_night_owl(person):
    total=0
    night_count=0
    for m in messages:
        if m['type'] not in('real', 'media', 'deleted'):
            continue
        if m['sender'] != person:
            continue
        total=total+1

        time_part=m['timestamp'].split(',',1)[1]
        clean_time=""
        for ch in time_part:
            if ch.isdigit() or ch == ':':
                clean_time=clean_time+ch
        hour=int(clean_time.split(':')[0])

        if hour >= 23 or hour <= 4:
            night_count=night_count+1

    if total==0:
        return 0
    return (night_count/total)*100

#==============================================================
#Archetype 4 : THE STORYTELLER - avg words per message > 30
#==============================================================
def score_storyteller(person):
    texts=person_messages.get(person,[])
    if len(texts) == 0:
        return 0
    total_words=0
    for text in texts:
        total_words=total_words+len(text.split())
    return total_words/len(texts)

#=====================================================================
#Archetype 5 : THE DRAMA QUEEN - % ALL-CAPS or 2+ exclamation marks
#=====================================================================
def score_drama_queen(person):
    texts=person_messages.get(person,[])
    if len(texts) == 0:
       return 0
    drama_count=0
    for text in texts:
        if len(text) < 3:
           continue
        exclaim_count=text.count('!')
        if text.isupper() or exclaim_count >= 2:
            drama_count=drama_count+1
    return (drama_count/len(texts))*100

#============================================================
#Archetype 6 : THE GHOST - silent on more than 60% of days
#============================================================
def score_ghost(person):
    active_dates=set()
    for m in messages:
        if m['type'] not in('real', 'media', 'deleted'):
            continue
        if m['sender'] != person:
            continue
        date_part=m['timestamp'].split(',',1)[0]
        active_dates.add(date_part)

    silent_days=total_days-len(active_dates)
    return (silent_days/total_days)*100

#=========================================================================
#Archetype 7 : THE COMEDIAN - %  messages with lol/lmao/haha/rofl/lmfao
#=========================================================================
def score_comedian(person):
    texts=person_messages.get(person,[])
    if len(texts) == 0:
        return 0
    funny_words=['lol','lmao','haha','rofl','lmfao']
    funny_count=0
    for text in texts:
        text_lower=text.lower()
        for word in funny_words:
            if word in text_lower:
               funny_count=funny_count+1
               break
    return(funny_count/len(texts))*100

#===============================================================
#Archetype 8 : THE QUESTION MASTER - % messages ending in '?'
#===============================================================
def score_question_master(person):
    texts=person_messages.get(person,[])
    if len(texts) == 0:
        return 0
    question_count=0
    for text in texts:
        if text.strip().endswith('?'):
           question_count=question_count+1
    return (question_count/len(texts))*100

#Step 2 : Compute all scores for every person
archetype_functions={
    'THE SPAMMER':score_spammer,
    'THE GROUP MOM':score_group_mom,
    'THE NIGHT OWL':score_night_owl,
    'THE STORYTELLER':score_storyteller,
    'THE DRAMA QUEEN':score_drama_queen,
    'THE GHOST':score_ghost,
    'THE COMEDIAN':score_comedian,
    'THE QUESTION MASTER':score_question_master
}

#Thresholds : only archetypes that clear their rule qualify(else fallback)
thresholds = {
    'THE SPAMMER': 3,
    'THE GROUP MOM': 0,      #highest count wins,no fixed threshold
    'THE NIGHT OWL': 60,
    'THE STORYTELLER': 30,
    'THE DRAMA QUEEN': 30,
    'THE GHOST': 60,
    'THE COMEDIAN': 0,
    'THE QUESTION MASTER': 25,
}

all_scores={}  #{person:{archetype:raw_score}}
for person in participants:
    all_scores[person]={}
    for archetype,func in archetype_functions.items():
        all_scores[person][archetype]=func(person)

#Step 3 : Assign exclusive archetypes
#Rule : sort people by their best-qualifying score,assign highest first,
#so no two people get the same archetype unless nobody else qualifies for it.
assigned_archetype={}
used_archetypes=set()

#Build a list of (person,archetype,score) for archetypes that clear threshold
candidates=[]
for person in participants:
    for archetype in archetype_functions:
        score=all_scores[person][archetype]
        if score >= thresholds[archetype]:
            candidates.append((person,archetype,score))

#Sort by score descending,so strongest matches get assigned first
candidates=sorted(candidates,key=lambda x:x[2],reverse=True)

for person,archetype,score in candidates:
    if person in assigned_archetype:
        continue
    if archetype in used_archetypes:
        continue
    assigned_archetype[person]=(archetype,score)
    used_archetypes.add(archetype)

#Fallback : anyone not yet assigned gets their single highest-scoring archetype
for person in participants:
    if person not in assigned_archetype:
       best_archetype=max(all_scores[person],key=all_scores[person].get)
       assigned_archetype[person]=(best_archetype,all_scores[person][best_archetype])

#Step 4 : Print the results
print("PERSONALITY ARCHETYPES")
for person in participants:
    archetype,score=assigned_archetype[person]
    print(f"{person}->{archetype} (score:{score:.1f})")

PERSONALITY ARCHETYPES
 Rahul->THE SPAMMER (score:4.5)
 Priya->THE GROUP MOM (score:605.0)
 Karan->THE STORYTELLER (score:57.0)
 Neha->THE DRAMA QUEEN (score:63.3)
 Aman->THE NIGHT OWL (score:79.8)
 Vikas->THE GHOST (score:73.3)


**Day 7 - Polish, Edge Cases:**



Feature 8 - The Final Report:**

Combine all previous features into one clean, formatted printed report using dividers, aligned columns, and clear sections.

In [29]:
#Feature 8 : The Final Report
#Combines everything into one clean,formatted printed report.

print("=" * 60)
print(" GROUPDNA REPORT - \"Hostel Bois 4ever\"")
print(f" {total_days} days • {total_messages} messages • {total_participants} members")
print("=" *60)

print(f" Period       : {first_date.strftime('%d %B %Y')} to {last_date.strftime('%d %B %Y')}")
print(f" Busiest day  : {busiest_day_readable} ({busiest_day_count} messages)")
print(f" Busiest hour : ({busiest_hour_range} ({busiest_hour_count} messages)")

print()
print("MESSAGES PER PERSON")
for person, count in sorted_people:
    percentage=(count/total_messages)*100
    bar_length=int((count / sorted_people[0][1]) * 20)
    bar='█' * bar_length
    print(f" {person:<8}{bar} {count} ({percentage:.1f}%)")

print()
print("ACTIVITY HEATMAP (hour of day, columns 00 to 23)")
print("          00  03  06  09  12  15  18  21")
for i in range(num_people):
    person=participants[i]
    row=heatmap[i]
    person_max=row.max()
    line=f"{person:<8}"
    for block_start in range(0, 24, 3):
        block_sum=row[block_start:block_start+3].sum()
        if person_max==0:
           symbol=". "
        else:
           ratio=block_sum/(person_max*3)
           if ratio <= 0.25:
               symbol=".  "
           elif ratio <= 0.50:
               symbol="░  "
           elif ratio <= 0.75:
               symbol="▒  "
           else:
               symbol="█  "
        line=line+symbol
    print(line)

print()
print(" THIS GROUP'S FAVOURITE WORDS")
word_max=top_10_words[0][1]
for word, count in top_10_words:
    bar_length=int((count/word_max)*20)
    bar='█' * bar_length
    print(f" {word:<10}{bar} {count}")

print()
print("RESPONSE PATTERNS")
print(f"Fastest replier: {fastest_person} (avg{avg_response_minutes[fastest_person]:.1f}minutes)")
if avg_response_minutes[slowest_person] >= 60:
    print(f"Slowest replier: {slowest_person} (avg{avg_response_minutes[slowest_person]/60:.1f} hours)")
else:
    print(f"Slowest replier: {slowest_person} (avg{avg_response_minutes[slowest_person]:.1f} minutes)")

print()
print("LONGEST SILENT STREAKS")
sorted_streaks=sorted(silent_streaks.items(),key=lambda x:x[1][0],reverse=True)
for person,(streak,start,end) in sorted_streaks:
    if streak == 0:
       print(f"{person:<8}: 0 days (never went silent)")
    else:
       print(f"{person:<8}: {streak} days ({start.strftime('%d %b')}-{end.strftime('%d %b')})")

print()
print("PERSONALITY ARCHETYPES")
for person in participants:
    archetype,score=assigned_archetype[person]
    print(f"{person}->{archetype} (score:{score:.1f})")

print()
print("=" * 60)
print("Generated by GroupDNA • Built with Python + NumPy")
print("=" * 60)

 GROUPDNA REPORT - "Hostel Bois 4ever"
 60 days • 3174 messages • 6 members
 Period       : 01 April 2024 to 30 May 2024
 Busiest day  : 04 May 2024 (76 messages)
 Busiest hour : (18.00-19.00 (248 messages)

MESSAGES PER PERSON
  Rahul  ████████████████████ 953 (30.0%)
  Priya  ███████████████ 718 (22.6%)
  Neha   █████████████ 635 (20.0%)
  Aman   ██████████ 490 (15.4%)
  Karan  ███████ 354 (11.2%)
  Vikas   24 (0.8%)

ACTIVITY HEATMAP (hour of day, columns 00 to 23)
          00  03  06  09  12  15  18  21
 Rahul  .  .  .  .  ░  ▒  ▒  ▒  
 Priya  .  .  ░  █  █  ▒  ▒  ░  
 Karan  .  .  .  ░  █  ▒  ▒  ░  
 Neha   .  .  ░  ▒  ▒  ▒  █  ░  
 Aman   ▒  ▒  .  .  .  .  .  ░  
 Vikas  .  .  ░  .  ░  ▒  ▒  ░  

 THIS GROUP'S FAVOURITE WORDS
 guys      ████████████████████ 318
 hai       ████████████████ 268
 telling   ███████████ 179
 bhai      ██████████ 160
 scene     █████████ 145
 yaar      ████████ 139
 kya       ████████ 133
 sleep     ███████ 112
 ja        ██████ 101
 raha      ██████ 